# NumPy Deep Dive — Solutions

**Course:** ML, Deep Learning & Computer Vision  

---

> Only review after you've made a genuine attempt at every exercise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

---
## Section 1 — Broadcasting

### 1.1 Predict the shape

In [ ]:
pairs = [
    ((5, 3), (3,)),         # (5, 3)
    ((5, 3), (5, 1)),       # (5, 3)
    ((5, 3), (1, 3)),       # (5, 3)
    ((5, 3), (5,)),         # ERROR — 3 vs 5
    ((2, 3, 4), (3, 4)),    # (2, 3, 4)
    ((2, 3, 4), (2, 1, 4)), # (2, 3, 4)
    ((2, 3, 4), (4,)),      # (2, 3, 4)
    ((2, 3, 4), (3, 1)),    # (2, 3, 4)
    ((6, 1, 4), (1, 5, 1)), # (6, 5, 4)
    ((3,), (4,)),           # ERROR — 3 vs 4
    ((1,), (5, 3)),         # (5, 3)
    ((8, 1, 6, 1), (7, 1, 5)),  # (8, 7, 6, 5)
]

for a_shape, b_shape in pairs:
    try:
        result = np.broadcast_shapes(a_shape, b_shape)
        print(f"{str(a_shape):16s} + {str(b_shape):16s} → {result}")
    except ValueError as e:
        print(f"{str(a_shape):16s} + {str(b_shape):16s} → ERROR: {e}")

### 1.2 Row-wise and column-wise operations

In [ ]:
X = np.random.randn(5, 4)
print("X:\n", X.round(3))

# a) Centre each row: row mean shape (5, 1)
a = X - X.mean(axis=1, keepdims=True)
print("\na) Row-centred means:", a.mean(axis=1).round(10))  # ≈ 0

# b) Centre each feature: col mean shape (1, 4) via keepdims
b = X - X.mean(axis=0, keepdims=True)
print("b) Col-centred means:", b.mean(axis=0).round(10))  # ≈ 0

# c) Standardise each feature
c = (X - X.mean(axis=0, keepdims=True)) / X.std(axis=0, keepdims=True)
print("c) Standardised stds:", c.std(axis=0).round(10))  # ≈ 1

# d) L2-normalise each row
norms = np.linalg.norm(X, axis=1, keepdims=True)  # (5, 1)
d = X / norms
print("d) Row norms after:", np.linalg.norm(d, axis=1).round(10))  # ≈ 1

### 1.3 Outer operations

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30])

# Reshape a to (4, 1) for broadcasting with (3,) → (4, 3)
print("a) Outer product:\n", a[:, None] * b)
print("b) Outer addition:\n", a[:, None] + b)
print("c) Outer abs diff:\n", np.abs(a[:, None] - b))
print("d) Outer max:\n", np.maximum(a[:, None], b))

### 1.4 Batch matrix operations

In [ ]:
batch, n, d, k = 3, 5, 8, 4
X = np.random.randn(batch, n, d)
w = np.random.randn(d)
W = np.random.randn(d, k)
b_vec = np.random.randn(batch, 1, d)

# a) Weighted sum: (batch, n, d) * (d,) → sum over d → (batch, n)
a = (X * w).sum(axis=-1)
print("a) shape:", a.shape)  # (3, 5)

# b) Linear layer: (batch, n, d) @ (d, k) → (batch, n, k)
b = X @ W
print("b) shape:", b.shape)  # (3, 5, 4)

# c) Add bias: (batch, n, d) + (batch, 1, d) → (batch, n, d)
c = X + b_vec
print("c) shape:", c.shape)  # (3, 5, 8)

# d) Per-batch mean: mean over n axis → (batch, d)
d_result = X.mean(axis=1)
print("d) shape:", d_result.shape)  # (3, 8)

# e) Per-batch cosine similarity
norms = np.linalg.norm(X, axis=-1, keepdims=True)  # (batch, n, 1)
X_normed = X / norms
cos_sim = X_normed @ X_normed.transpose(0, 2, 1)  # (batch, n, n)
print("e) shape:", cos_sim.shape)  # (3, 5, 5)
print("   diagonal (should be 1):", cos_sim[0].diagonal().round(4))

### 1.5 Attention scores

In [ ]:
batch, seq_len, d_k, d_v = 2, 6, 8, 8
Q = np.random.randn(batch, seq_len, d_k)
K = np.random.randn(batch, seq_len, d_k)
V = np.random.randn(batch, seq_len, d_v)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

# 1. Raw scores
scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)  # (batch, seq, seq)

# 2. Causal mask: upper triangle = -inf
mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)  # True above diagonal
scores[:, mask] = -np.inf

# 3. Softmax
weights = softmax(scores)  # (batch, seq, seq)

# 4. Output
output = weights @ V  # (batch, seq, d_v)

print("Scores shape:", scores.shape)
print("Weights shape:", weights.shape)
print("Output shape:", output.shape)
print("\nWeights row sums (should be 1):", weights[0].sum(axis=-1).round(6))
print("\nWeights[0] (notice zeros above diagonal):")
print(weights[0].round(3))

---
## Section 2 — Advanced indexing

### 2.1 One-hot encoding

In [ ]:
def one_hot(labels, n_classes):
    result = np.zeros((len(labels), n_classes), dtype=int)
    result[np.arange(len(labels)), labels] = 1
    return result

labels = np.array([0, 2, 1, 0, 3, 2])
result = one_hot(labels, 4)
print(result)
assert result.shape == (6, 4)
assert np.all(result.sum(axis=1) == 1)
print("Passed!")

### 2.2 Embedding lookup

In [ ]:
vocab_size, embed_dim = 1000, 64
embedding_matrix = np.random.randn(vocab_size, embed_dim)
token_ids = np.array([[5, 102, 47, 0], [88, 3, 999, 42]])

# One line — fancy indexing broadcasts over the batch
embeddings = embedding_matrix[token_ids]

print("Shape:", embeddings.shape)  # (2, 4, 64)
assert embeddings.shape == (2, 4, 64)
assert np.allclose(embeddings[0, 0], embedding_matrix[5])  # verify first lookup
print("Passed!")

### 2.3 Cross-entropy loss

In [ ]:
def cross_entropy_loss(logits, targets):
    # Numerically stable softmax
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    # Gather the probability of the correct class
    correct_probs = probs[np.arange(len(targets)), targets]
    return -np.log(correct_probs).mean()

logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.3],
                   [1.0, 1.0, 3.0]])
targets = np.array([0, 1, 2])

loss = cross_entropy_loss(logits, targets)
print(f"Loss: {loss:.4f}")  # ~0.3068

### 2.4 Padding and masking

In [ ]:
sequences = [
    np.array([4, 7, 2]),
    np.array([1, 9, 3, 5, 8]),
    np.array([6, 2]),
    np.array([3, 1, 7, 4]),
]

# a) Pad
max_len = max(len(s) for s in sequences)
padded = np.zeros((len(sequences), max_len), dtype=int)
for i, s in enumerate(sequences):
    padded[i, :len(s)] = s
print("Padded:\n", padded)

# b) Mask (vectorised: lengths array + arange broadcasting)
lengths = np.array([len(s) for s in sequences])  # [3, 5, 2, 4]
mask = np.arange(max_len) < lengths[:, None]     # (4, 1) vs (5,) → (4, 5)
print("\nMask:\n", mask)

# c) Mean of real values (mask broadcasting)
# Replace padding with 0 (already is), sum real values, divide by real count
masked_sum = (padded * mask).sum(axis=1)
means = masked_sum / lengths
print("\nMeans:", means.round(4))

# Verify manually: first row [4, 7, 2] → mean = 4.333
print("Check row 0:", np.array([4, 7, 2]).mean())

### 2.5 Top-k per row

In [ ]:
def topk_per_row(arr, k):
    # argsort descending per row
    sorted_idx = np.argsort(-arr, axis=1)[:, :k]  # (n, k)
    # Gather values using fancy indexing
    rows = np.arange(arr.shape[0])[:, None]  # (n, 1) for broadcasting
    sorted_vals = arr[rows, sorted_idx]       # (n, k)
    return sorted_idx, sorted_vals

np.random.seed(42)
data = np.random.randn(4, 8)
print("Data:\n", data.round(2))

idx, vals = topk_per_row(data, 3)
print("\nTop-3 indices:\n", idx)
print("Top-3 values:\n", vals.round(2))
assert np.all(np.diff(vals, axis=1) <= 0)
print("Passed!")

---
## Section 3 — Vectorisation challenges

### 3.1 Pairwise squared distances

In [ ]:
import time

A = np.random.randn(100, 16)
B = np.random.randn(80, 16)

# Method 1: Expand and subtract
t = time.time()
D1 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)
t1 = time.time() - t

# Method 2: Algebraic trick ||a-b||² = ||a||² + ||b||² - 2a·b
t = time.time()
a_sq = (A ** 2).sum(axis=1, keepdims=True)  # (100, 1)
b_sq = (B ** 2).sum(axis=1, keepdims=True).T  # (1, 80)
D2 = a_sq + b_sq - 2 * A @ B.T
D2 = np.maximum(D2, 0)  # numerical stability
t2 = time.time() - t

# Method 3: einsum
t = time.time()
diff = A[:, None, :] - B[None, :, :]
D3 = np.einsum('ijk,ijk->ij', diff, diff)
t3 = time.time() - t

# Verify
print("Method 1 vs 2 match:", np.allclose(D1, D2, atol=1e-10))
print("Method 1 vs 3 match:", np.allclose(D1, D3, atol=1e-10))
print(f"\nTimings: expand={t1:.5f}s  algebraic={t2:.5f}s  einsum={t3:.5f}s")

### 3.2 2D convolution

In [ ]:
def conv2d(image, kernel):
    kH, kW = kernel.shape
    windows = np.lib.stride_tricks.sliding_window_view(image, (kH, kW))
    return (windows * kernel).sum(axis=(-2, -1))

# Checkerboard
img = np.zeros((64, 64))
img[::8, :] = 1; img[:, ::8] = 1

horiz_edge = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float)
vert_edge = horiz_edge.T
gaussian = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=float) / 16

results = [
    ("Original", img, None),
    ("Horizontal edges", conv2d(img, horiz_edge), "RdBu_r"),
    ("Vertical edges", conv2d(img, vert_edge), "RdBu_r"),
    ("Gaussian blur", conv2d(img, gaussian), "gray"),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (title, data, cmap) in zip(axes, results):
    ax.imshow(data, cmap=cmap or "gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.3 Batch normalisation

In [ ]:
def batch_norm(X, gamma, beta, eps=1e-5):
    mean = X.mean(axis=(0, 2, 3), keepdims=True)     # (1, C, 1, 1)
    var = X.var(axis=(0, 2, 3), keepdims=True)        # (1, C, 1, 1)
    X_hat = (X - mean) / np.sqrt(var + eps)
    return gamma * X_hat + beta

batch, channels, H, W = 8, 3, 16, 16
X = np.random.randn(batch, channels, H, W) * 5 + 10
gamma = np.ones((1, channels, 1, 1))
beta = np.zeros((1, channels, 1, 1))

Y = batch_norm(X, gamma, beta)
print("Input  — per-channel mean:", X.mean(axis=(0, 2, 3)).round(2))
print("Input  — per-channel std: ", X.std(axis=(0, 2, 3)).round(2))
print("Output — per-channel mean:", Y.mean(axis=(0, 2, 3)).round(4))
print("Output — per-channel std: ", Y.std(axis=(0, 2, 3)).round(4))

### 3.4 Non-max suppression

In [ ]:
def compute_iou_matrix(boxes):
    x1 = boxes[:, 0]; y1 = boxes[:, 1]
    x2 = boxes[:, 2]; y2 = boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    # Intersection: broadcast (N,1) vs (1,N)
    inter_x1 = np.maximum(x1[:, None], x1[None, :])
    inter_y1 = np.maximum(y1[:, None], y1[None, :])
    inter_x2 = np.minimum(x2[:, None], x2[None, :])
    inter_y2 = np.minimum(y2[:, None], y2[None, :])
    inter_area = np.maximum(0, inter_x2 - inter_x1) * np.maximum(0, inter_y2 - inter_y1)
    union_area = areas[:, None] + areas[None, :] - inter_area
    return inter_area / union_area

def nms(boxes, scores, iou_threshold=0.5):
    iou = compute_iou_matrix(boxes)
    order = scores.argsort()[::-1]
    keep = []
    suppressed = np.zeros(len(boxes), dtype=bool)
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        suppressed |= iou[i] > iou_threshold
        suppressed[i] = False  # don't suppress self
    return keep

boxes = np.array([
    [100, 100, 210, 210],
    [105, 108, 215, 215],
    [300, 300, 400, 400],
    [102, 102, 212, 212],
    [305, 305, 405, 405],
], dtype=float)
scores = np.array([0.9, 0.75, 0.95, 0.8, 0.6])

print("IoU matrix:")
print(compute_iou_matrix(boxes).round(2))

kept = nms(boxes, scores, iou_threshold=0.5)
print("\nKept indices:", kept)

### 3.5 Image mosaic

In [ ]:
def make_grid(images):
    N, H, W, C = images.shape
    ncols = int(np.ceil(np.sqrt(N)))
    nrows = int(np.ceil(N / ncols))
    # Pad to fill the grid if needed
    pad_n = nrows * ncols - N
    if pad_n > 0:
        padding = np.zeros((pad_n, H, W, C), dtype=images.dtype)
        images = np.concatenate([images, padding], axis=0)
    return images.reshape(nrows, ncols, H, W, C).transpose(0, 2, 1, 3, 4).reshape(nrows * H, ncols * W, C)

np.random.seed(42)
images = np.random.randint(0, 256, (16, 32, 32, 3), dtype=np.uint8)
grid = make_grid(images)
print("Grid shape:", grid.shape)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(grid)
ax.set_title("4×4 Image Grid (no loops)")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Section 4 — Memory & performance

### 4.1 Views vs copies

In [ ]:
a = np.arange(20).reshape(4, 5)

tests = [
    ("Basic slice",     lambda a: a[1:3, 2:4],    "VIEW"),
    ("Fancy indexing",  lambda a: a[[0, 2], :],    "COPY"),
    ("Boolean indexing",lambda a: a[a > 10],       "COPY"),
    ("Transpose",       lambda a: a.T,             "VIEW"),
    ("Reshape",         lambda a: a.reshape(2,10), "VIEW"),
    ("Flatten",         lambda a: a.flatten(),     "COPY"),
    ("Ravel",           lambda a: a.ravel(),       "VIEW"),
]

for name, fn, expected in tests:
    b = fn(a)
    is_view = np.shares_memory(a, b)
    status = "VIEW" if is_view else "COPY"
    check = "✓" if status == expected else "✗"
    print(f"{check} {name:20s} → {status} (expected {expected})")

### 4.2 Sliding window via stride tricks

In [ ]:
arr = np.random.randint(0, 100, (8, 8))
print("Array:\n", arr)

# a) 3×3 windows
windows = np.lib.stride_tricks.sliding_window_view(arr, (3, 3))
print("\nWindows shape:", windows.shape)  # (6, 6, 3, 3)

# b) Mean of each window
window_means = windows.mean(axis=(-2, -1))
print("Window means shape:", window_means.shape)

# c) Highest-sum window
window_sums = windows.sum(axis=(-2, -1))
max_pos = np.unravel_index(window_sums.argmax(), window_sums.shape)
print(f"Highest-sum window at top-left ({max_pos[0]}, {max_pos[1]}), sum = {window_sums[max_pos]}")

# d) Verify view
print("Is view (shares memory):", np.shares_memory(arr, windows))

### 4.3 Memory-efficient operations

In [ ]:
import time

X_orig = np.random.randn(10000, 1000)
print(f"X size: {X_orig.nbytes / 1e6:.1f} MB")

# a) Naive
X = X_orig.copy()
t = time.time()
mean = X.mean(axis=0)
std = X.std(axis=0)
centred = X - mean
result_a = centred / std
ta = time.time() - t

# b) In-place
X = X_orig.copy()
t = time.time()
mean = X.mean(axis=0)
std = X.std(axis=0)
X -= mean
X /= std
result_b = X
tb = time.time() - t

# c) With out=
X = X_orig.copy()
t = time.time()
mean = X.mean(axis=0)
std = X.std(axis=0)
np.subtract(X, mean, out=X)
np.divide(X, std, out=X)
result_c = X
tc = time.time() - t

print(f"\nNaive:    {ta:.4f}s")
print(f"In-place: {tb:.4f}s")
print(f"out=:     {tc:.4f}s")
print(f"\nAll match: {np.allclose(result_a, result_b) and np.allclose(result_a, result_c)}")

---
## Section 5 — Capstone: K-means

In [ ]:
def kmeans(X, k, max_iter=50, tol=1e-6):
    n, d = X.shape
    # Init centroids: random subset of points
    idx = np.random.choice(n, k, replace=False)
    centroids = X[idx].copy()
    inertia_history = []

    for it in range(max_iter):
        # Assign: pairwise sq distances (algebraic trick)
        x_sq = (X ** 2).sum(axis=1, keepdims=True)   # (n, 1)
        c_sq = (centroids ** 2).sum(axis=1, keepdims=True).T  # (1, k)
        dists = x_sq + c_sq - 2 * X @ centroids.T    # (n, k)
        labels = dists.argmin(axis=1)                 # (n,)

        # Inertia = sum of min distances
        inertia = dists[np.arange(n), labels].sum()
        inertia_history.append(inertia)

        # Update centroids
        new_centroids = np.zeros_like(centroids)
        for c in range(k):
            mask = labels == c
            if mask.any():
                new_centroids[c] = X[mask].mean(axis=0)
            else:
                new_centroids[c] = centroids[c]  # keep if empty

        # Check convergence
        shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        if shift < tol:
            print(f"Converged at iteration {it + 1}")
            break

    return labels, centroids, inertia_history

# Generate data
np.random.seed(42)
centres = np.array([[2, 2], [-2, 2], [2, -2], [-2, -2]])
X = np.vstack([c + np.random.randn(100, 2) * 0.6 for c in centres])

# Run
labels, centroids, inertia = kmeans(X, k=4)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ["#378ADD", "#1D9E75", "#D85A30", "#7F77DD"]
for c in range(4):
    mask = labels == c
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[c], s=15, alpha=0.6, label=f"Cluster {c}")
axes[0].scatter(centroids[:, 0], centroids[:, 1], c="black", marker="X", s=150, zorder=5, label="Centroids")
axes[0].set_title("K-means clustering")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.2)

axes[1].plot(inertia, color="#378ADD", linewidth=2)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Inertia")
axes[1].set_title("Convergence")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
**End of solutions.** If you got through all of these without loops, your NumPy skills are production-ready.